# Let's build a data agent

The data covers World Cups from 1930 to 2026: 11 related tables (matches, teams, players, goals, bookings, lineups, substitutions, referees, match_referees, penalty_shootouts, tournaments) in a local DuckDB file.


## Part 1 — Set up the workshop

`smolagents` supplies the agent loop, LiteLLM connects it to Amazon Bedrock, and DuckDB is the warehouse.

In [1]:
# The workshop environment is preconfigured in SageMaker.

In [2]:
import re
from pathlib import Path
from bedrock import build_bedrock_model
from notebook_ui import run_agent
import duckdb
from smolagents import tool
from agent import ToolCallingAgent

### Connect to the supplied database

The supplied World Cup database is already beside this notebook.

In [3]:
DB_PATH = Path("worldcups-1930-2026.duckdb")
assert DB_PATH.exists(), "The supplied World Cup database is missing."

# Read-only at the connection level, not merely in the prompt.
con = duckdb.connect(str(DB_PATH), read_only=True)
TABLES = [row[0] for row in con.execute("SHOW TABLES").fetchall()]

print("database:", DB_PATH.resolve())
print("tables:", TABLES)
print("matches:", con.execute("SELECT count(*) FROM matches").fetchone()[0])

database: /Users/pavlinam/dev/confidence/data-agent-workshop-eval-suite/worldcups-1930-2026.duckdb
tables: ['bookings', 'goals', 'lineups', 'match_referees', 'matches', 'penalty_shootouts', 'players', 'referees', 'substitutions', 'teams', 'tournaments']
matches: 1068


### Configure the model

Press Enter to use the default, **Claude Haiku 4.5**. We start with a fast, cost-efficient model for the live exercises. In Part 5, we will compare it with stronger models using the same tools, instructions, and questions.


In [4]:
model = build_bedrock_model()

Bedrock API key (input hidden; paste and press Enter):  ········
AWS region [eu-north-1]:  
Bedrock model [bedrock/eu.anthropic.claude-haiku-4-5-20251001-v1:0]:  


model: bedrock/eu.anthropic.claude-haiku-4-5-20251001-v1:0


## Part 2 — A minimal agent (and its first lie)

Two tools only: list the tables, run SQL. No schema documentation, no business rules, no guardrails. First it will genuinely impress you. Then it won't.


In [5]:
MAX_ROWS = 50

@tool
def list_tables() -> str:
    """List the tables available in the World Cup warehouse."""
    return "\n".join(TABLES)

@tool
def execute_sql(query: str) -> str:
    """Call this tool to execute SQL.

    Args:
        query: One SELECT query.
    """
    try:
        df = con.execute(query).df()
    except Exception:
        return "Query failed"
#    except Exception as exc:
#        return f"ERROR: {type(exc).__name__}: {exc}"
    if df.empty:
        return "Query returned 0 rows."
    return df.head(MAX_ROWS).to_markdown(index=False)

In [6]:
MINIMAL_PROMPT = """You are a data analyst answering from a DuckDB warehouse.
Use your tools to answer questions. Keep the final answer concise."""
def make_agent(tools, instructions, max_steps=20, agent_model=None):
    return ToolCallingAgent(
        tools=tools,
        model=agent_model or model,
        instructions=instructions,
        max_steps=max_steps,
    )
minimal_agent = make_agent([list_tables, execute_sql], MINIMAL_PROMPT)

### Try out some questions

- How many matches in the 2026 world cup?
- How many goals have been scored in World Cup history?
- Who won the 2026 World Cup and what was the final score (To get this right, you need to modify the agent a bit)


### Watch the trace, not only the final answer:
- Replay the same questions multiple times. Are they consistent?
- How many steps did it use?
- Did a failed query give it enough information to recover?
- Could a valid SQL give a wrong answer?
- Did it know the schema, or guess column names?


In [7]:
run_agent(minimal_agent, "Who won the 2026 World Cup and what was the final score")

## Part 3 — Read the schema first, add a compact data profiler

The agent goes straight into querying, gets errors, realizes it needs to explore the schema. Let's help it by adding a specialized tool to get the schema, nulls, and a few actual values. It avoids dumping entire tables into the context window.
The agent doesn't have to discover it all on it's own through failures.

In [8]:
def _quoted_table(name: str) -> str:
    if name not in TABLES:
        raise ValueError(f"Unknown table {name!r}. Available: {', '.join(TABLES)}")
    return '"' + name.replace('"', '""') + '"'

@tool
def inspect_table(table_name: str) -> str:
    """Profile one table: columns, types, null counts, and example values.

    Args:
        table_name: Exact table name returned by list_tables.
    """
    try:
        table = _quoted_table(table_name)
    except ValueError as exc:
        return f"ERROR: {exc}"
    row_count = con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    columns = con.execute(f"DESCRIBE {table}").fetchall()
    lines = [f"table: {table_name}", f"rows: {row_count}", "columns:"]
    for name, dtype, *_ in columns:
        column = '"' + name.replace('"', '""') + '"'
        nulls = con.execute(
            f"SELECT count(*) FILTER (WHERE {column} IS NULL) FROM {table}"
        ).fetchone()[0]
        examples = con.execute(
            f"SELECT DISTINCT {column} FROM {table} WHERE {column} IS NOT NULL LIMIT 4"
        ).fetchall()
        sample = ", ".join(repr(row[0]) for row in examples)
        lines.append(f"- {name}: {dtype}; nulls={nulls}; examples=[{sample}]")
    return "\n".join(lines)

### Try out some questions

Try again the previous question. Should be a smoother flow now.  
- Who won the 2026 World Cup and what was the final score?

Then lets try this one, seems like an easy one. 

- Which player has played the most world cups? 

In [ ]:
PROFILE_PROMPT = MINIMAL_PROMPT + """
Before querying, use the inspect table tool to inspect every relevant table. Use observed column names and values.
Resolve internal IDs to human-readable names before answering."""

profile_agent = make_agent(
    [list_tables, inspect_table, execute_sql],
    PROFILE_PROMPT,
)


run_agent(profile_agent, "Which player has played the most world cups?")


### A schema profile still does not explain data semantics

The profiler run is *efficient* — a couple of inspect calls and clean queries — but it may claim that Luis Suarez played seven tournaments. That is not true: the Uruguayan Luis Suarez played four, while two other players share his name. The schema alone cannot explain that distinction, so the agent needs business context.


## Part 4 — Add a data contract

Critical semantics are short enough to place directly in the system instructions. Grouping by name alone merges different footballers; grouping by name and team preserves the identity distinction encoded by this dataset.

In [ ]:
DATA_CONTRACT = """
World Cup warehouse contract

Relationships
- every match belongs to one tournament: matches.tournament_id ->
  tournaments.tournament_id, and tournaments.season is the year (e.g. 2026)
- matches.home_team_id and away_team_id -> teams.team_id
- players.team_id -> teams.team_id
- goals.player_id -> players.player_id
- goals.credited_team_id -> teams.team_id
- lineups, bookings, and penalty_shootouts use match_id, team_id, and player_id
- substitutions use match_id and team_id, with player_on_id and player_off_id
  both pointing to players

Data Guidance
- Player identity and careers
- One players row is one World Cup appearance: one person, one tournament,
  one team. The warehouse stores appearances; it has no table for humans.
- Names are not people. The same name on different national teams is
  different people, and the same name in eras far apart is different
  people. One person keeps one name across all their tournaments - even
  when their country is renamed, a renamed country is still the same team.
- Career questions count tournaments per person: group appearances by
  name and team together (a name on two different teams is two people),
  and compare people - never names.

Join safety
- goals, bookings, lineups, substitutions, and penalty_shootouts are fact tables.
  Aggregate each fact to the required grain before joining facts together.
"""

CONTEXT_PROMPT = PROFILE_PROMPT + "\n" + DATA_CONTRACT
context_agent = make_agent(
    [list_tables, inspect_table, execute_sql],
    CONTEXT_PROMPT,
)

In [ ]:
run_agent(context_agent, "which player has played the most world cups?")


- Lionel MESSI and CRISTIANO RONALDO with 6 tournaments is the correct answer.



## Part 5 — Compare models with an eval suite

A convincing demo is not enough: a model change can improve quality, but it can also increase latency and cost. We will hold the tools and data-contract prompt constant while comparing **Claude Haiku 4.5**, **Claude Sonnet 4.5**, and **Claude Sonnet 4.6**. **Claude Opus 4.5** is the independent judge.

The judge compares each answer with a reference answer. The harness also records steps, tool calls, latency, tokens, and model cost when LiteLLM has pricing for the model. Only the candidate model changes; tools, instructions, scenarios, and execution order stay the same. Candidates and scenarios run sequentially so they can safely share the notebook's DuckDB connection.

In [ ]:
from evaluation import EvalSuite, build_bedrock_judge, build_candidate_models

judge_model = build_bedrock_judge()
candidate_models = build_candidate_models()
candidate_agents = {
    name: make_agent(
        [list_tables, inspect_table, execute_sql],
        CONTEXT_PROMPT,
        agent_model=candidate_model,
    )
    for name, candidate_model in candidate_models.items()
}

eval_suite = EvalSuite(
    candidates=candidate_agents,
    judge_model=judge_model,
)

# A deliberately simple workshop API. This shadows Python's built-in eval().
eval = eval_suite.eval

### Demo scenario 1 — identity semantics

This catches the mistake where several people with the same name are merged into one career.

In [ ]:
eval(
    question="Which player has played in the most World Cups?",
    expected_answer=(
        "Cristiano Ronaldo of Portugal and Lionel Messi of Argentina are tied "
        "for the most, with 6 World Cups each, including 2026."
    ),
    criteria="Do not merge different people merely because their names match.",
    scenario="player identity",
)

### Demo scenario 2 — safe joins between fact tables

This catches fan-out: joining bookings directly to goals multiplies rows and produces convincing but incorrect totals.

In [ ]:
eval(
    question=(
        "At the 2022 World Cup, which team received the most bookings, and how "
        "many goals did that same team score?"
    ),
    expected_answer="Argentina received 17 bookings and scored 15 goals.",
    criteria=(
        "Count booking rows and goal rows separately before combining the totals. "
        "The answer must include the team and both counts."
    ),
    scenario="fact-table fan-out",
)

### Compare the complete suite

Pass rate shows quality; the remaining columns show its operational price. A missing cost means LiteLLM has no price entry for that model, while tokens, latency, and steps are still available.

In [ ]:
eval_suite.summary()

### Add your own scenario

Copy this shape. The optional criteria should explain any business rule that the judge must enforce. Every call is added to the suite summary.

```python
eval(
    question="Your data question",
    expected_answer="The reference answer",
    criteria="Optional scoring requirements",
    scenario="short scenario name",
)
```

For a real regression suite, add several representative scenarios and rerun them after every prompt, tool, schema, or model change. LLM judges are useful but not infallible, so review reference answers and periodically audit judge decisions.